# AI Workforce Capacity Planning Platform
## Notebook 00 — Project Setup and Enterprise Parameter Framework

**Implementation:** 05 — Enterprise Parameter Framework  
**Project version:** 2.1.0  
**Notebook version:** 3.0.0  

This notebook is the shared platform bootstrap for all downstream notebooks.

It centralizes:

1. project identity,
2. environment configuration,
3. persistent S3 paths,
4. pipeline defaults,
5. forecasting defaults and runtime limits,
6. model-selection defaults,
7. capacity-planning defaults,
8. AI-assistant defaults,
9. reusable parameter validation,
10. storage-access validation.

**Compatibility contract:** Existing global constants remain available so the
validated `02_data_pipeline` notebook continues to work without modification.

In [0]:
from __future__ import annotations

from datetime import datetime, timezone
from typing import Any, Mapping, Sequence

## Section 01 — Project Identity

In [0]:
PROJECT_NAME = "AI Workforce Capacity Planning Platform"
PROJECT_KEY = "overtime-capacity-planning"
PROJECT_VERSION = "2.1.0"
PROJECT_SETUP_VERSION = "3.0.0"
ENVIRONMENT = "development"
PROJECT_OWNER = "Issouf KABRE"

PROJECT_INITIALIZED_AT_UTC = datetime.now(timezone.utc)

PROJECT_CONFIG: dict[str, Any] = {
    "project_name": PROJECT_NAME,
    "project_key": PROJECT_KEY,
    "project_version": PROJECT_VERSION,
    "project_setup_version": PROJECT_SETUP_VERSION,
    "environment": ENVIRONMENT,
    "project_owner": PROJECT_OWNER,
}

## Section 02 — Persistent Storage Configuration

Bronze remains the first persistent data layer.
Source files may exist temporarily during acquisition, but no persistent
Landing copy is required for the current prototype architecture.

In [0]:
S3_BUCKET = "issouf-data-lake"
PROJECT_ROOT = f"s3a://{S3_BUCKET}/{PROJECT_KEY}"

BRONZE_ROOT = f"{PROJECT_ROOT}/bronze"
SILVER_ROOT = f"{PROJECT_ROOT}/silver"
GOLD_ROOT = f"{PROJECT_ROOT}/gold"

METADATA_ROOT = f"{PROJECT_ROOT}/metadata"
REGISTRY_ROOT = f"{PROJECT_ROOT}/registry"
MANIFEST_ROOT = f"{METADATA_ROOT}/manifests"
VALIDATION_ROOT = f"{METADATA_ROOT}/validation"
PIPELINE_LOG_ROOT = f"{METADATA_ROOT}/pipeline_logs"

MODEL_ROOT = f"{PROJECT_ROOT}/models"
FORECAST_ROOT = f"{PROJECT_ROOT}/forecasts"
DECISION_ROOT = f"{PROJECT_ROOT}/decisions"
REPORT_ROOT = f"{PROJECT_ROOT}/reports"

DATASET_REGISTRY_PATH = f"{REGISTRY_ROOT}/dataset_registry"

STORAGE_CONFIG: dict[str, str] = {
    "s3_bucket": S3_BUCKET,
    "project_root": PROJECT_ROOT,
    "bronze_root": BRONZE_ROOT,
    "silver_root": SILVER_ROOT,
    "gold_root": GOLD_ROOT,
    "metadata_root": METADATA_ROOT,
    "registry_root": REGISTRY_ROOT,
    "dataset_registry_path": DATASET_REGISTRY_PATH,
    "manifest_root": MANIFEST_ROOT,
    "validation_root": VALIDATION_ROOT,
    "pipeline_log_root": PIPELINE_LOG_ROOT,
    "model_root": MODEL_ROOT,
    "forecast_root": FORECAST_ROOT,
    "decision_root": DECISION_ROOT,
    "report_root": REPORT_ROOT,
}

# Backward-compatible shared path map.
PROJECT_PATHS: dict[str, str] = {
    "project_root": PROJECT_ROOT,
    "bronze": BRONZE_ROOT,
    "silver": SILVER_ROOT,
    "gold": GOLD_ROOT,
    "metadata": METADATA_ROOT,
    "registry": REGISTRY_ROOT,
    "dataset_registry": DATASET_REGISTRY_PATH,
    "manifests": MANIFEST_ROOT,
    "validation": VALIDATION_ROOT,
    "pipeline_logs": PIPELINE_LOG_ROOT,
    "models": MODEL_ROOT,
    "forecasts": FORECAST_ROOT,
    "decisions": DECISION_ROOT,
    "reports": REPORT_ROOT,
}

In [0]:
# ============================================================
# Enterprise Metadata Storage Configuration
#
# Unity Catalog Volumes are used for metadata persistence,
# governance artifacts, and demonstration datasets.
#
# Persistent analytical datasets continue to reside in S3.
# ============================================================

PROJECT_VOLUME = "/Volumes/dev/default/project_storage"

ENTERPRISE_METADATA_ROOT = (
    f"{PROJECT_VOLUME}/enterprise_metadata"
)

METADATA_CATALOG_PATH = (
    f"{ENTERPRISE_METADATA_ROOT}/catalog"
)

SAMPLE_DATASET_PATH = (
    f"{ENTERPRISE_METADATA_ROOT}/sample_dataset"
)

UNITY_CATALOG_STORAGE_CONFIG: dict[str, str] = {
    "project_volume": PROJECT_VOLUME,
    "enterprise_metadata_root": ENTERPRISE_METADATA_ROOT,
    "metadata_catalog_path": METADATA_CATALOG_PATH,
    "sample_dataset_path": SAMPLE_DATASET_PATH,
}

print("=" * 60)
print("Enterprise Metadata Storage Configuration")
print("=" * 60)
print(f"Project Volume       : {PROJECT_VOLUME}")
print(f"Metadata Root        : {ENTERPRISE_METADATA_ROOT}")
print(f"Metadata Catalog     : {METADATA_CATALOG_PATH}")
print(f"Sample Dataset       : {SAMPLE_DATASET_PATH}")
print("=" * 60)

### 2.1 Enterprise Metadata Storage Configuration  

Unity Catalog volume paths support governed metadata catalog persistence  
without changing the platform's existing S3 data-lake architecture.

## Section 03 — Data-Pipeline Parameters

In [0]:
PIPELINE_CONFIG: dict[str, Any] = {
    "pipeline_name": "enterprise-workforce-data-foundation",
    "pipeline_version": "3.1.0",
    "write_mode": "overwrite",
    "parquet_compression": "snappy",
    "enable_data_quality_checks": True,
    "save_execution_log": True,
    "fail_on_empty_dataset": True,
    "fail_on_row_count_mismatch": True,
    "fail_on_duplicate_business_keys": True,
}

# Backward-compatible constants for downstream notebooks.
PIPELINE_NAME = PIPELINE_CONFIG["pipeline_name"]
PIPELINE_VERSION = PIPELINE_CONFIG["pipeline_version"]
PARQUET_WRITE_MODE = PIPELINE_CONFIG["write_mode"]
PARQUET_COMPRESSION = PIPELINE_CONFIG["parquet_compression"]

## Section 04 — Forecast Parameters

The active forecast horizon is a runtime parameter.

This configuration defines:

- a default value,
- accepted boundaries,
- validation behavior,
- forecasting metadata.

It does **not** hard-code the business to one-day, seven-day, or
fourteen-day forecasting.

In [0]:
FORECAST_CONFIG: dict[str, Any] = {
    "default_horizon_days": 14,
    "minimum_horizon_days": 1,
    "maximum_horizon_days": 90,
    "frequency": "daily",
    "date_column": "order_date",
    "target_column": "workload_units",
    "validation_horizon_days": 28,
    "minimum_training_rows": 180,
    "random_seed": 42,
    "retrain_model": True,
    "save_model": True,
    "confidence_level": 0.95,
}

DEFAULT_FORECAST_HORIZON_DAYS = FORECAST_CONFIG["default_horizon_days"]
MINIMUM_FORECAST_HORIZON_DAYS = FORECAST_CONFIG["minimum_horizon_days"]
MAXIMUM_FORECAST_HORIZON_DAYS = FORECAST_CONFIG["maximum_horizon_days"]

## Section 05 — Model Parameters

In [0]:
MODEL_CONFIG: dict[str, Any] = {
    "candidate_models": [
        "seasonal_naive",
        "linear_regression",
        "random_forest",
        "gradient_boosting",
    ],
    "primary_metric": "mae",
    "secondary_metrics": [
        "rmse",
        "mape",
        "wape",
    ],
    "selection_strategy": "lowest_validation_mae",
    "model_artifact_format": "mlflow",
    "register_best_model": True,
}

## Section 06 — Capacity-Planning Parameters

These values are defaults for the public-data prototype.
Operational production values must be confirmed with warehouse
stakeholders before real-world deployment.

In [0]:
# Capacity-planning defaults for the public-data prototype.
# Production values must be validated with warehouse stakeholders.

CAPACITY_CONFIG: dict[str, Any] = {
    # Normal scheduled capacity
    "standard_shift_hours": 10.0,
    "maximum_daily_hours": 10.0,

    # Workforce productivity measure
    "productivity_unit": "order_lines_per_associate_hour",

    # Capacity-gap decision thresholds
    "voluntary_capacity_gap_ratio": 0.50,
    "mandatory_capacity_gap_ratio": 1.00,
}


OVERTIME_POLICY: dict[str, Any] = {
    # Supported overtime classifications
    "voluntary_enabled": True,
    "mandatory_enabled": True,

    # Overtime is performed outside the normal scheduled workday.
    "allowed_day_types": (
        "OFF_DAY",
        "SATURDAY",
        "SUNDAY",
    ),

    # Calendar-policy controls
    "weekend_enabled": True,
    "holiday_adjustment_enabled": True,

    # Overtime shift boundaries
    "minimum_shift_hours": 5.0,
    "maximum_shift_hours": 10.0,
}


SLA_CONFIG: dict[str, Any] = {
    # SLA clock starts when workload is released into Manhattan.
    "processing_commitment_hours": 48,

    # Capacity and productivity are measured using order lines.
    "workload_unit": "order_lines",

    # Out-of-cycle workload consideration
    "include_current_oc_backlog": True,
    "include_projected_oc_backlog": True,

    # Supported SLA risk classifications
    "risk_levels": (
        "LOW",
        "MEDIUM",
        "HIGH",
        "CRITICAL",
    ),
}


PLANNING_HORIZON_CONFIG: dict[str, Any] = {
    # Operational and tactical visibility
    "next_day_days": 1,
    "weekly_days": 7,

    # Strategic visibility
    "monthly_days": 30,
    "quarterly_days": 90,

    # Runtime boundaries
    "minimum_horizon_days": 1,
    "maximum_horizon_days": 90,
}

## Section 07 — AI-Assistant Parameters

Secrets and credentials must never be stored in this configuration.

In [0]:
AI_CONFIG: dict[str, Any] = {
    # Feature control
    "assistant_enabled": True,
    "human_review_required": True,

    # Context included in AI responses
    "include_forecast_context": True,
    "include_capacity_context": True,
    "include_sla_context": True,
    "include_oc_backlog_context": True,
    "include_overtime_policy_context": True,
    "include_decision_explanation": True,

    # Supported planning horizons
    "supported_horizons_days": (
        1,
        7,
        30,
        90,
    ),

    # Response controls
    "maximum_context_records": 30,
    "response_style": "operations_management",
    "require_evidence_summary": True,
    "require_confidence_statement": True,

    # Safety and governance
    "allow_autonomous_workforce_decisions": False,
    "allow_autonomous_overtime_scheduling": False,
}

## Section 08 — Shared Validation Utilities

Reusable validation functions for configuration dictionaries, numeric  
parameters, runtime horizons, and controlled option values.

In [0]:
def validate_required_keys(
    *,
    config_name: str,
    config: Mapping[str, Any],
    required_keys: Sequence[str],
) -> None:
    """Validate that a configuration contains all required non-null keys."""

    if not isinstance(config, Mapping):
        raise TypeError(
            f"{config_name} must be a mapping, "
            f"received {type(config).__name__}."
        )

    missing_keys = [
        key
        for key in required_keys
        if key not in config or config[key] is None
    ]

    if missing_keys:
        raise ValueError(
            f"{config_name} is missing required keys: "
            f"{sorted(missing_keys)}"
        )


def validate_integer_range(
    *,
    parameter_name: str,
    value: Any,
    minimum: int,
    maximum: int,
) -> int:
    """Validate that a value is an integer within an inclusive range."""

    if isinstance(value, bool) or not isinstance(value, int):
        raise TypeError(
            f"{parameter_name} must be an integer, "
            f"received {type(value).__name__}."
        )

    if minimum > maximum:
        raise ValueError(
            f"Invalid validation range for {parameter_name}: "
            f"minimum {minimum} exceeds maximum {maximum}."
        )

    if not minimum <= value <= maximum:
        raise ValueError(
            f"{parameter_name} must be between "
            f"{minimum} and {maximum}; received {value}."
        )

    return value


def validate_numeric_range(
    *,
    parameter_name: str,
    value: Any,
    minimum: float,
    maximum: float,
) -> float:
    """Validate that a numeric value is within an inclusive range."""

    if isinstance(value, bool) or not isinstance(value, (int, float)):
        raise TypeError(
            f"{parameter_name} must be numeric, "
            f"received {type(value).__name__}."
        )

    numeric_value = float(value)

    if minimum > maximum:
        raise ValueError(
            f"Invalid validation range for {parameter_name}: "
            f"minimum {minimum} exceeds maximum {maximum}."
        )

    if not minimum <= numeric_value <= maximum:
        raise ValueError(
            f"{parameter_name} must be between "
            f"{minimum} and {maximum}; received {numeric_value}."
        )

    return numeric_value


def validate_non_empty_string(
    *,
    parameter_name: str,
    value: Any,
) -> str:
    """Validate that a value is a non-empty string."""

    if not isinstance(value, str):
        raise TypeError(
            f"{parameter_name} must be a string, "
            f"received {type(value).__name__}."
        )

    normalized_value = value.strip()

    if not normalized_value:
        raise ValueError(f"{parameter_name} cannot be empty.")

    return normalized_value


def validate_string_choice(
    *,
    parameter_name: str,
    value: Any,
    allowed_values: Sequence[str],
) -> str:
    """Validate that a string belongs to a controlled set of values."""

    normalized_value = validate_non_empty_string(
        parameter_name=parameter_name,
        value=value,
    )

    allowed_set = set(allowed_values)

    if not allowed_set:
        raise ValueError(
            f"No allowed values were configured for {parameter_name}."
        )

    if normalized_value not in allowed_set:
        raise ValueError(
            f"{parameter_name} must be one of "
            f"{sorted(allowed_set)}; received {normalized_value!r}."
        )

    return normalized_value


def validate_boolean(
    *,
    parameter_name: str,
    value: Any,
) -> bool:
    """Validate that a configuration value is Boolean."""

    if not isinstance(value, bool):
        raise TypeError(
            f"{parameter_name} must be Boolean, "
            f"received {type(value).__name__}."
        )

    return value


def validate_non_empty_sequence(
    *,
    parameter_name: str,
    value: Any,
) -> Sequence[Any]:
    """Validate a non-empty list or tuple configuration value."""

    if isinstance(value, (str, bytes)) or not isinstance(
        value,
        Sequence,
    ):
        raise TypeError(
            f"{parameter_name} must be a list or tuple, "
            f"received {type(value).__name__}."
        )

    if len(value) == 0:
        raise ValueError(f"{parameter_name} cannot be empty.")

    return value

## Section 09 — Runtime Parameter Contract  

Resolve the active forecast horizon while preserving configured minimum,  
default, and maximum planning boundaries.

In [0]:
def resolve_forecast_horizon(
    requested_horizon: int | None = None,
) -> int:
    """
    Resolve and validate the active forecast horizon.

    The configured default is used when no runtime override is supplied.
    """

    horizon = (
        FORECAST_CONFIG["default_horizon_days"]
        if requested_horizon is None
        else requested_horizon
    )

    return validate_integer_range(
        parameter_name="forecast_horizon_days",
        value=horizon,
        minimum=FORECAST_CONFIG["minimum_horizon_days"],
        maximum=FORECAST_CONFIG["maximum_horizon_days"],
    )


ACTIVE_FORECAST_HORIZON_DAYS = resolve_forecast_horizon()


RUNTIME_CONFIG: dict[str, Any] = {
    "forecast_horizon_days": ACTIVE_FORECAST_HORIZON_DAYS,
    "forecast_frequency": FORECAST_CONFIG["frequency"],
    "environment": ENVIRONMENT,
    "initialized_at_utc": PROJECT_INITIALIZED_AT_UTC,
}

## Section 10 — Platform Configuration Validation  

Validate all centralized platform contracts before downstream notebooks  
consume the shared configuration.

In [0]:
def validate_platform_configuration() -> None:
    """Validate all shared platform configuration contracts."""

    validate_required_keys(
        config_name="PROJECT_CONFIG",
        config=PROJECT_CONFIG,
        required_keys=[
            "project_name",
            "project_key",
            "project_version",
            "project_setup_version",
            "environment",
            "project_owner",
        ],
    )

    validate_required_keys(
        config_name="STORAGE_CONFIG",
        config=STORAGE_CONFIG,
        required_keys=[
            "project_root",
            "bronze_root",
            "silver_root",
            "gold_root",
            "metadata_root",
            "registry_root",
            "dataset_registry_path",
            "manifest_root",
            "validation_root",
            "pipeline_log_root",
            "model_root",
            "forecast_root",
            "decision_root",
            "report_root",
        ],
    )

    validate_required_keys(
        config_name="PIPELINE_CONFIG",
        config=PIPELINE_CONFIG,
        required_keys=[
            "pipeline_name",
            "pipeline_version",
            "write_mode",
            "parquet_compression",
            "enable_data_quality_checks",
            "save_execution_log",
        ],
    )

    validate_required_keys(
        config_name="FORECAST_CONFIG",
        config=FORECAST_CONFIG,
        required_keys=[
            "default_horizon_days",
            "minimum_horizon_days",
            "maximum_horizon_days",
            "frequency",
            "date_column",
            "target_column",
            "validation_horizon_days",
            "minimum_training_rows",
            "confidence_level",
        ],
    )

    validate_required_keys(
        config_name="MODEL_CONFIG",
        config=MODEL_CONFIG,
        required_keys=[
            "candidate_models",
            "primary_metric",
            "secondary_metrics",
            "selection_strategy",
            "model_artifact_format",
        ],
    )

    validate_required_keys(
        config_name="CAPACITY_CONFIG",
        config=CAPACITY_CONFIG,
        required_keys=[
            "standard_shift_hours",
            "maximum_daily_hours",
            "productivity_unit",
            "voluntary_capacity_gap_ratio",
            "mandatory_capacity_gap_ratio",
        ],
    )

    validate_required_keys(
        config_name="OVERTIME_POLICY",
        config=OVERTIME_POLICY,
        required_keys=[
            "voluntary_enabled",
            "mandatory_enabled",
            "allowed_day_types",
            "weekend_enabled",
            "holiday_adjustment_enabled",
            "minimum_shift_hours",
            "maximum_shift_hours",
        ],
    )

    validate_required_keys(
        config_name="SLA_CONFIG",
        config=SLA_CONFIG,
        required_keys=[
            "processing_commitment_hours",
            "workload_unit",
            "include_current_oc_backlog",
            "include_projected_oc_backlog",
            "risk_levels",
        ],
    )

    validate_required_keys(
        config_name="PLANNING_HORIZON_CONFIG",
        config=PLANNING_HORIZON_CONFIG,
        required_keys=[
            "next_day_days",
            "weekly_days",
            "monthly_days",
            "quarterly_days",
            "minimum_horizon_days",
            "maximum_horizon_days",
        ],
    )

    validate_required_keys(
        config_name="AI_CONFIG",
        config=AI_CONFIG,
        required_keys=[
            "assistant_enabled",
            "human_review_required",
            "supported_horizons_days",
            "maximum_context_records",
            "response_style",
            "allow_autonomous_workforce_decisions",
            "allow_autonomous_overtime_scheduling",
        ],
    )

    # Project contract
    validate_non_empty_string(
        parameter_name="project_name",
        value=PROJECT_CONFIG["project_name"],
    )
    validate_non_empty_string(
        parameter_name="project_key",
        value=PROJECT_CONFIG["project_key"],
    )
    validate_non_empty_string(
        parameter_name="environment",
        value=PROJECT_CONFIG["environment"],
    )

    # Pipeline contract
    validate_string_choice(
        parameter_name="pipeline_write_mode",
        value=PIPELINE_CONFIG["write_mode"],
        allowed_values=("overwrite", "append", "error", "ignore"),
    )

    validate_non_empty_string(
        parameter_name="parquet_compression",
        value=PIPELINE_CONFIG["parquet_compression"],
    )

    validate_boolean(
        parameter_name="enable_data_quality_checks",
        value=PIPELINE_CONFIG["enable_data_quality_checks"],
    )

    validate_boolean(
        parameter_name="save_execution_log",
        value=PIPELINE_CONFIG["save_execution_log"],
    )

    # Forecast horizon contract
    minimum_horizon = validate_integer_range(
        parameter_name="minimum_horizon_days",
        value=FORECAST_CONFIG["minimum_horizon_days"],
        minimum=1,
        maximum=365,
    )

    default_horizon = validate_integer_range(
        parameter_name="default_horizon_days",
        value=FORECAST_CONFIG["default_horizon_days"],
        minimum=1,
        maximum=365,
    )

    maximum_horizon = validate_integer_range(
        parameter_name="maximum_horizon_days",
        value=FORECAST_CONFIG["maximum_horizon_days"],
        minimum=1,
        maximum=365,
    )

    if not minimum_horizon <= default_horizon <= maximum_horizon:
        raise ValueError(
            "FORECAST_CONFIG requires "
            "minimum_horizon_days <= default_horizon_days "
            "<= maximum_horizon_days."
        )

    validate_integer_range(
        parameter_name="validation_horizon_days",
        value=FORECAST_CONFIG["validation_horizon_days"],
        minimum=1,
        maximum=maximum_horizon,
    )

    validate_integer_range(
        parameter_name="minimum_training_rows",
        value=FORECAST_CONFIG["minimum_training_rows"],
        minimum=1,
        maximum=10_000_000,
    )

    validate_numeric_range(
        parameter_name="confidence_level",
        value=FORECAST_CONFIG["confidence_level"],
        minimum=0.50,
        maximum=0.999,
    )

    # Model contract
    validate_non_empty_sequence(
        parameter_name="candidate_models",
        value=MODEL_CONFIG["candidate_models"],
    )

    validate_non_empty_sequence(
        parameter_name="secondary_metrics",
        value=MODEL_CONFIG["secondary_metrics"],
    )

    validate_non_empty_string(
        parameter_name="primary_metric",
        value=MODEL_CONFIG["primary_metric"],
    )

    # Capacity contract
    standard_shift_hours = validate_numeric_range(
        parameter_name="standard_shift_hours",
        value=CAPACITY_CONFIG["standard_shift_hours"],
        minimum=1.0,
        maximum=24.0,
    )

    maximum_daily_hours = validate_numeric_range(
        parameter_name="maximum_daily_hours",
        value=CAPACITY_CONFIG["maximum_daily_hours"],
        minimum=1.0,
        maximum=24.0,
    )

    if standard_shift_hours > maximum_daily_hours:
        raise ValueError(
            "Standard shift hours cannot exceed maximum daily hours."
        )

    voluntary_gap_ratio = validate_numeric_range(
        parameter_name="voluntary_capacity_gap_ratio",
        value=CAPACITY_CONFIG["voluntary_capacity_gap_ratio"],
        minimum=0.0,
        maximum=10.0,
    )

    mandatory_gap_ratio = validate_numeric_range(
        parameter_name="mandatory_capacity_gap_ratio",
        value=CAPACITY_CONFIG["mandatory_capacity_gap_ratio"],
        minimum=0.0,
        maximum=10.0,
    )

    if voluntary_gap_ratio >= mandatory_gap_ratio:
        raise ValueError(
            "The voluntary capacity-gap ratio must be lower than "
            "the mandatory capacity-gap ratio."
        )

    # Overtime contract
    validate_boolean(
        parameter_name="voluntary_overtime_enabled",
        value=OVERTIME_POLICY["voluntary_enabled"],
    )

    validate_boolean(
        parameter_name="mandatory_overtime_enabled",
        value=OVERTIME_POLICY["mandatory_enabled"],
    )

    allowed_day_types = validate_non_empty_sequence(
        parameter_name="allowed_overtime_day_types",
        value=OVERTIME_POLICY["allowed_day_types"],
    )

    for day_type in allowed_day_types:
        validate_non_empty_string(
            parameter_name="overtime_day_type",
            value=day_type,
        )

    minimum_overtime_hours = validate_numeric_range(
        parameter_name="minimum_overtime_shift_hours",
        value=OVERTIME_POLICY["minimum_shift_hours"],
        minimum=1.0,
        maximum=24.0,
    )

    maximum_overtime_hours = validate_numeric_range(
        parameter_name="maximum_overtime_shift_hours",
        value=OVERTIME_POLICY["maximum_shift_hours"],
        minimum=1.0,
        maximum=24.0,
    )

    if minimum_overtime_hours > maximum_overtime_hours:
        raise ValueError(
            "Minimum overtime shift hours cannot exceed maximum "
            "overtime shift hours."
        )

    if maximum_overtime_hours > maximum_daily_hours:
        raise ValueError(
            "Maximum overtime shift hours cannot exceed the configured "
            "maximum daily hours."
        )

    # SLA contract
    validate_integer_range(
        parameter_name="processing_commitment_hours",
        value=SLA_CONFIG["processing_commitment_hours"],
        minimum=1,
        maximum=720,
    )

    validate_non_empty_string(
        parameter_name="sla_workload_unit",
        value=SLA_CONFIG["workload_unit"],
    )

    validate_boolean(
        parameter_name="include_current_oc_backlog",
        value=SLA_CONFIG["include_current_oc_backlog"],
    )

    validate_boolean(
        parameter_name="include_projected_oc_backlog",
        value=SLA_CONFIG["include_projected_oc_backlog"],
    )

    risk_levels = validate_non_empty_sequence(
        parameter_name="sla_risk_levels",
        value=SLA_CONFIG["risk_levels"],
    )

    for risk_level in risk_levels:
        validate_non_empty_string(
            parameter_name="sla_risk_level",
            value=risk_level,
        )

    # Planning-horizon alignment
    planning_minimum = validate_integer_range(
        parameter_name="planning_minimum_horizon_days",
        value=PLANNING_HORIZON_CONFIG["minimum_horizon_days"],
        minimum=1,
        maximum=365,
    )

    planning_maximum = validate_integer_range(
        parameter_name="planning_maximum_horizon_days",
        value=PLANNING_HORIZON_CONFIG["maximum_horizon_days"],
        minimum=1,
        maximum=365,
    )

    if planning_minimum != minimum_horizon:
        raise ValueError(
            "Planning and forecast minimum horizons must match."
        )

    if planning_maximum != maximum_horizon:
        raise ValueError(
            "Planning and forecast maximum horizons must match."
        )

    configured_horizons = {
        validate_integer_range(
            parameter_name="next_day_days",
            value=PLANNING_HORIZON_CONFIG["next_day_days"],
            minimum=planning_minimum,
            maximum=planning_maximum,
        ),
        validate_integer_range(
            parameter_name="weekly_days",
            value=PLANNING_HORIZON_CONFIG["weekly_days"],
            minimum=planning_minimum,
            maximum=planning_maximum,
        ),
        validate_integer_range(
            parameter_name="monthly_days",
            value=PLANNING_HORIZON_CONFIG["monthly_days"],
            minimum=planning_minimum,
            maximum=planning_maximum,
        ),
        validate_integer_range(
            parameter_name="quarterly_days",
            value=PLANNING_HORIZON_CONFIG["quarterly_days"],
            minimum=planning_minimum,
            maximum=planning_maximum,
        ),
    }

    # AI governance contract
    validate_boolean(
        parameter_name="assistant_enabled",
        value=AI_CONFIG["assistant_enabled"],
    )

    validate_boolean(
        parameter_name="human_review_required",
        value=AI_CONFIG["human_review_required"],
    )

    validate_integer_range(
        parameter_name="maximum_context_records",
        value=AI_CONFIG["maximum_context_records"],
        minimum=1,
        maximum=10_000,
    )

    supported_horizons = set(
        validate_non_empty_sequence(
            parameter_name="supported_horizons_days",
            value=AI_CONFIG["supported_horizons_days"],
        )
    )

    if supported_horizons != configured_horizons:
        raise ValueError(
            "AI-supported horizons must match the configured "
            "planning horizons."
        )

    if not AI_CONFIG["human_review_required"]:
        raise ValueError(
            "Human review must remain required for workforce decisions."
        )

    if AI_CONFIG["allow_autonomous_workforce_decisions"]:
        raise ValueError(
            "Autonomous workforce decisions are prohibited."
        )

    if AI_CONFIG["allow_autonomous_overtime_scheduling"]:
        raise ValueError(
            "Autonomous overtime scheduling is prohibited."
        )


validate_platform_configuration()
CONFIGURATION_STATUS = "PASSED"

## Section 11 — Storage Access Validation  

Verify that required persistent S3 locations are accessible before a  
downstream pipeline begins execution.

In [0]:
def validate_storage_access(
    paths: Mapping[str, str],
) -> dict[str, str]:
    """
    Validate configured storage paths using lightweight directory access.

    Returns a path-by-path validation result. Any inaccessible path causes
    the notebook to fail before downstream processing begins.
    """

    if not isinstance(paths, Mapping):
        raise TypeError("Storage paths must be provided as a mapping.")

    if not paths:
        raise ValueError("No storage paths were supplied for validation.")

    results: dict[str, str] = {}
    failures: list[str] = []

    for path_name, path_value in paths.items():
        path = validate_non_empty_string(
            parameter_name=f"storage_path[{path_name}]",
            value=path_value,
        )

        try:
            # mkdirs is idempotent and confirms write/access capability.
            dbutils.fs.mkdirs(path)

            # ls confirms that Databricks can resolve the location.
            dbutils.fs.ls(path)

            results[path_name] = "ACCESSIBLE"

        except Exception as exc:
            results[path_name] = f"FAILED: {type(exc).__name__}"
            failures.append(
                f"{path_name}={path}: {type(exc).__name__}: {exc}"
            )

    if failures:
        failure_details = "\n".join(failures)
        raise RuntimeError(
            "Storage-access validation failed:\n"
            f"{failure_details}"
        )

    return results


STORAGE_VALIDATION_PATHS: dict[str, str] = {
    "project_root": PROJECT_ROOT,
    "bronze_root": BRONZE_ROOT,
    "silver_root": SILVER_ROOT,
    "gold_root": GOLD_ROOT,
    "metadata_root": METADATA_ROOT,
    "registry_root": REGISTRY_ROOT,
    "manifest_root": MANIFEST_ROOT,
    "validation_root": VALIDATION_ROOT,
    "pipeline_log_root": PIPELINE_LOG_ROOT,
    "model_root": MODEL_ROOT,
    "forecast_root": FORECAST_ROOT,
    "decision_root": DECISION_ROOT,
    "report_root": REPORT_ROOT,
}


STORAGE_VALIDATION_RESULTS = validate_storage_access(
    STORAGE_VALIDATION_PATHS
)

STORAGE_STATUS = "PASSED"

# Backward-compatible Boolean used by downstream notebooks.
STORAGE_CONNECTION_OK = STORAGE_STATUS == "PASSED"

RUNTIME_STATUS = "READY"

## Section 12 — Execution Summary  

Publish the validated platform bootstrap status for operators and  
downstream notebooks.

In [0]:
SETUP_EXECUTION_SUMMARY: dict[str, Any] = {
    "project_name": PROJECT_NAME,
    "project_key": PROJECT_KEY,
    "project_version": PROJECT_VERSION,
    "project_setup_version": PROJECT_SETUP_VERSION,
    "environment": ENVIRONMENT,
    "project_root": PROJECT_ROOT,
    "active_forecast_horizon_days": (
        ACTIVE_FORECAST_HORIZON_DAYS
    ),
    "configuration_status": CONFIGURATION_STATUS,
    "storage_status": STORAGE_STATUS,
    "runtime_status": RUNTIME_STATUS,
    "initialized_at_utc": (
        PROJECT_INITIALIZED_AT_UTC.isoformat()
    ),
}


print("=" * 72)
print("AI WORKFORCE CAPACITY PLANNING PLATFORM")
print("PROJECT SETUP EXECUTION SUMMARY")
print("=" * 72)
print(f"Project name         : {PROJECT_NAME}")
print(f"Project key          : {PROJECT_KEY}")
print(f"Project version      : {PROJECT_VERSION}")
print(f"Setup version        : {PROJECT_SETUP_VERSION}")
print(f"Environment          : {ENVIRONMENT}")
print(f"Project root         : {PROJECT_ROOT}")
print(
    "Forecast horizon    : "
    f"{ACTIVE_FORECAST_HORIZON_DAYS} day(s)"
)
print(f"Configuration status : {CONFIGURATION_STATUS}")
print(f"Storage status       : {STORAGE_STATUS}")
print(f"Runtime status       : {RUNTIME_STATUS}")
print("=" * 72)